In [9]:
# ============================================================
# ULB Credit Card Fraud — Preprocessing Step 1: Diagnostics
# ============================================================
import numpy as np
import pandas as pd
import os

notebook_dir = os.path.dirname(os.path.abspath('__file__'))
CSV_PATH =  os.path.join(notebook_dir, "..", "data", "raw", "creditcard.csv")   # <-- change to your local path

# ---------- 1. Load ----------
df = pd.read_csv(CSV_PATH)

print("=== SHAPE ===")
print(df.shape)
print("\n=== COLUMNS ===")
print(df.columns.tolist())
print("\n=== DTYPES ===")
print(df.dtypes.value_counts())
print("\n=== HEAD ===")
print(df.head())

# ---------- 2. Missing-value verification ----------
print("\n=== MISSING VALUES ===")
total_missing = df.isnull().sum().sum()
print("Total missing cells:", total_missing)

per_col = df.isnull().sum()
print("Columns with missing:", per_col[per_col > 0].to_dict() if total_missing else "NONE")

print("NaN cells:", int(df.isna().sum().sum()))
num_inf = int(np.isinf(df.select_dtypes(include=[np.number]).to_numpy()).sum())
print("Inf cells:", num_inf)

# ---------- 3. Class balance ----------
print("\n=== CLASS BALANCE ===")
print(df['Class'].value_counts())
print(df['Class'].value_counts(normalize=True).map('{:.6%}'.format))

# ---------- 4. Amount & Time descriptive stats ----------
print("\n=== AMOUNT (overall) ===")
print(df['Amount'].describe().round(4))
print("skewness:", round(df['Amount'].skew(), 4))
print("kurtosis:", round(df['Amount'].kurtosis(), 4))

print("\n=== AMOUNT by class ===")
print(df.groupby('Class')['Amount'].describe().round(4))

print("\n=== TIME (overall) ===")
print(df['Time'].describe().round(4))
print("skewness:", round(df['Time'].skew(), 4))

print("\n=== TIME by class ===")
print(df.groupby('Class')['Time'].describe().round(4))

# ---------- 5. Outlier analysis on Amount (IQR) ----------
def iqr_info(s, k=1.5):
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    lo, hi = q1 - k * iqr, q3 + k * iqr
    return lo, hi, iqr

lo, hi, iqr = iqr_info(df['Amount'])
out_mask = (df['Amount'] < lo) | (df['Amount'] > hi)

print("\n=== AMOUNT OUTLIERS (IQR, k=1.5) ===")
print(f"Q1, Q3, IQR: {df['Amount'].quantile(.25):.2f}, "
      f"{df['Amount'].quantile(.75):.2f}, {iqr:.2f}")
print(f"Lower/Upper bounds: [{lo:.2f}, {hi:.2f}]")
print(f"Total outliers: {out_mask.sum()} ({out_mask.mean()*100:.2f}%)")
print(f"  Among legit (Class=0): "
      f"{out_mask[df['Class']==0].sum()} "
      f"({out_mask[df['Class']==0].mean()*100:.2f}% of legit rows)")
print(f"  Among fraud (Class=1): "
      f"{out_mask[df['Class']==1].sum()} "
      f"({out_mask[df['Class']==1].mean()*100:.2f}% of fraud rows)")

# Mean vs median tells us about skew
print(f"\nAmount mean={df['Amount'].mean():.2f}, "
      f"median={df['Amount'].median():.2f}, "
      f"max={df['Amount'].max():.2f}")

# ---------- 6. Sanity check on V1–V28 (already PCA'd) ----------
v_cols = [c for c in df.columns if c.startswith('V')]
print("\n=== V1–V28 SANITY CHECK ===")
print("Count V columns:", len(v_cols))
print("Mean abs of V means:", round(np.abs(df[v_cols].mean()).mean(), 6))
print("Mean of V stds    :", round(df[v_cols].std().mean(), 6))
print("Min/Max V mean    :",
      round(df[v_cols].mean().min(), 6), "/", round(df[v_cols].mean().max(), 6))
print("Min/Max V std    :",
      round(df[v_cols].std().min(), 6), "/", round(df[v_cols].std().max(), 6))

=== SHAPE ===
(284807, 31)

=== COLUMNS ===
['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount', 'Class']

=== DTYPES ===
float64    30
int64       1
Name: count, dtype: int64

=== HEAD ===
   Time        V1        V2        V3        V4        V5        V6        V7  \
0   0.0 -1.359807 -0.072781  2.536347  1.378155 -0.338321  0.462388  0.239599   
1   0.0  1.191857  0.266151  0.166480  0.448154  0.060018 -0.082361 -0.078803   
2   1.0 -1.358354 -1.340163  1.773209  0.379780 -0.503198  1.800499  0.791461   
3   1.0 -0.966272 -0.185226  1.792993 -0.863291 -0.010309  1.247203  0.237609   
4   2.0 -1.158233  0.877737  1.548718  0.403034 -0.407193  0.095921  0.592941   

         V8        V9  ...       V21       V22       V23       V24       V25  \
0  0.098698  0.363787  ... -0.018307  0.277838 -0.110474  0.066928  0.128539   
1  0.08510

In [10]:
# ============================================================
# ULB Credit Card Fraud — Preprocessing Step 2: Scaling & Export
# ============================================================
import pandas as pd
from sklearn.preprocessing import RobustScaler
import os

CSV_PATH = os.path.join(notebook_dir, "..", "data", "raw", "creditcard.csv")
OUTPUT_PATH = os.path.join(notebook_dir, "..", "data", "processed", "creditcard_preprocessed.csv")

# 1. Load the dataset
df = pd.read_csv(CSV_PATH)

# 2. Initialize RobustScaler (uses median and IQR)
scaler = RobustScaler()

# 3. Apply RobustScaler to 'Amount' and 'Time'
cols_to_scale = ['Amount', 'Time']
df[cols_to_scale] = scaler.fit_transform(df[cols_to_scale])

# 4. Verify V1-V28 were not touched by only scaling the specified columns
# (Already handled by explicitly passing cols_to_scale to the scaler)

# 5. Save the preprocessed dataset
df.to_csv(OUTPUT_PATH, index=False)
print(f"Preprocessed dataset saved successfully to: {OUTPUT_PATH}")

# ---------- 6. Final Verification ----------
print("\n=== POST-SCALING VERIFICATION ===")

# Check for any NaNs introduced accidentally
print("Total NaNs after scaling:", int(df.isnull().sum().sum()))

# Verify Class balance is preserved
print("\nClass balance preserved:")
print(df['Class'].value_counts(normalize=True).map('{:.6%}'.format))

# Verify Amount and Time stats
print("\nScaled Amount & Time stats:")
print(df[cols_to_scale].describe().round(4))

# Verify V1-V28 are untouched
v_cols = [c for c in df.columns if c.startswith('V')]
print("\nV1-V28 check (should still be mean ~0, std ~1):")
print("Mean abs of V means:", round(abs(df[v_cols].mean()).mean(), 6))
print("Mean of V stds    :", round(df[v_cols].std().mean(), 6))

Preprocessed dataset saved successfully to: c:\Users\53542\Desktop\mplad-anomaly-detection\notebooks\..\data\processed\creditcard_preprocessed.csv

=== POST-SCALING VERIFICATION ===
Total NaNs after scaling: 0

Class balance preserved:
Class
0    99.827251%
1     0.172749%
Name: proportion, dtype: str

Scaled Amount & Time stats:
            Amount         Time
count  284807.0000  284807.0000
mean        0.9271       0.1189
std         3.4950       0.5579
min        -0.3074      -0.9950
25%        -0.2292      -0.3582
50%         0.0000       0.0000
75%         0.7708       0.6418
max       358.6832       1.0350

V1-V28 check (should still be mean ~0, std ~1):
Mean abs of V means: 0.0
Mean of V stds    : 0.976391
